# Part 4 — Edge Cases

This notebook walks through every common pitfall that trips people up when writing DataFrames to PostgreSQL — and shows that `PostgresConnector` handles each one automatically.

**Prerequisites:** [Part 1](Part1_Getting_Started.ipynb)

In [ ]:
import pandas as pd
import numpy as np
from postgres_connector import PostgresConnector

pg = PostgresConnector(
    host="localhost",
    database="my_db",
    username="postgres",
    password="mysecretpassword",
    schema="tutorial",
)
pg.execute_query("CREATE SCHEMA IF NOT EXISTS tutorial;")

## Edge Case 1 — NumPy Scalar Types

**Problem:** psycopg 2 raises `can't adapt type 'numpy.int64'` when you pass NumPy scalars as SQL parameters. This happens silently whenever Pandas infers `int64` or `float64` column types.

**What the connector does:** `_clean_records()` calls `.item()` on every NumPy scalar before it reaches the driver, converting it to a plain Python `int` or `float`.

In [ ]:
# Explicitly typed numpy scalars — exactly what Pandas produces after groupby, astype, etc.
df = pd.DataFrame({
    "id":       [np.int64(1), np.int64(2), np.int64(3)],
    "score":    [np.float64(9.5), np.float64(8.1), np.float64(7.3)],
    "rank":     np.array([1, 2, 3], dtype=np.int32),
})

print("Column dtypes:", df.dtypes.to_dict())

# This would crash with raw psycopg2 — works fine here
pg.upsert_data(df, "scores", primary_key="id")
print("\nInserted successfully.")
pg.get_data("SELECT * FROM scores;")

## Edge Case 2 — NaN, None, and NaT become NULL

**Problem:** Pandas uses `float('nan')` for missing numeric values and `pd.NaT` for missing timestamps. Neither maps directly to SQL `NULL` with a naive driver.

**What the connector does:** `_clean_records()` calls `pd.isna()` on every value and converts any truthy result to `None` before sending to the DB.

In [ ]:
df = pd.DataFrame({
    "id":          [1, 2, 3],
    "revenue":     [100.0, float("nan"), 300.0],   # NaN
    "category":    ["A", None, "C"],                # None
    "updated_at":  pd.to_datetime(["2024-01-01", pd.NaT, "2024-01-03"]),  # NaT
})

pg.upsert_data(df, "nulls_demo", primary_key="id")

# All three missing values stored as proper SQL NULL
result = pg.get_data("SELECT * FROM nulls_demo ORDER BY id;")
print(result)
print("\nNULL count per column:")
print(result.isna().sum())

## Edge Case 3 — Mixed-Case Column Names

**Problem:** PostgreSQL stores unquoted identifiers in lowercase. If your DataFrame has `Product_ID` or `REVENUE`, a naive write creates columns that you can never reliably query without double-quoting.

**What the connector does:** `_lower_df_columns()` lowercases every column name before any DDL or DML is run. The table is always created with lowercase columns.

In [ ]:
df = pd.DataFrame({
    "Product_ID":   [1, 2],
    "NAME":         ["Widget", "Gadget"],
    "SALE_Price":   [9.99, 19.99],
})
print("Original columns:", df.columns.tolist())

pg.upsert_data(df, "casing_demo", primary_key="product_id")

# Columns in DB are all lowercase — no quoting needed in queries
result = pg.get_data("SELECT product_id, name, sale_price FROM casing_demo;")
print("\nDB columns:", result.columns.tolist())
result

## Edge Case 4 — Empty DataFrame

**Problem:** Passing an empty DataFrame to a naive INSERT causes either a crash or a silent no-op that's hard to diagnose.

**What the connector does:** All three write methods (`upsert_data`, `replace_table`, `delete_and_insert`) check `df.empty` at the top and return immediately with no DB interaction.

In [ ]:
empty = pd.DataFrame(columns=["id", "value"])
print(f"Is empty: {empty.empty}")

# None of these raise — they just return immediately
pg.upsert_data(empty, "empty_test", primary_key="id")
pg.replace_table(empty, "empty_test")
pg.delete_and_insert(empty, "empty_test", delete_keys="id")

# Table was never created because nothing ran
print("Table exists:", pg.check_table_exists("empty_test"))

## Edge Case 5 — Very Wide DataFrames (32 767 Parameter Limit)

**Problem:** PostgreSQL rejects any statement that contains more than 32 767 bind parameters. A DataFrame with 100 columns and 500 rows = 50 000 parameters — an instant error.

**What the connector does:** `_safe_chunk_size()` computes the maximum safe batch size from the column count and automatically splits the DataFrame into chunks small enough to stay under the limit.

In [ ]:
from postgres_connector import PostgresConnector

# Simulate a very wide DataFrame: 200 columns × 1 000 rows
# Naively: 200 × 1000 = 200 000 parameters — well over the limit
wide_df = pd.DataFrame(
    np.random.rand(1_000, 200),
    columns=[f"metric_{i:03d}" for i in range(200)],
)
wide_df.insert(0, "id", range(1_000))

# Demonstrate what chunk size the library computes
n_cols = len(wide_df.columns)   # 201
chunk = PostgresConnector._safe_chunk_size(n_cols)
print(f"Columns: {n_cols}  →  Safe chunk size: {chunk} rows/batch")
print(f"Parameters per batch: {n_cols * chunk} (limit: 32 767)")

# This upsert silently splits into however many chunks are needed
pg.upsert_data(wide_df, "wide_table", primary_key="id")
count = pg.get_data("SELECT COUNT(*) AS n FROM wide_table;").iloc[0]["n"]
print(f"\nAll {count} rows inserted without error.")

## Edge Case 6 — Multi-Schema Isolation

**Problem:** Multiple teams sharing one PostgreSQL database can accidentally clobber each other's tables if everything lives in `public`.

**What the connector does:** Every DDL and DML statement is prefixed with the schema passed in the constructor. Two connectors pointing at different schemas never interfere — even if both use the same table name.

In [ ]:
pg_team_a = PostgresConnector(
    host="localhost", database="my_db",
    username="postgres", password="mysecretpassword",
    schema="team_a",
)
pg_team_b = PostgresConnector(
    host="localhost", database="my_db",
    username="postgres", password="mysecretpassword",
    schema="team_b",
)

pg_team_a.execute_query("CREATE SCHEMA IF NOT EXISTS team_a;")
pg_team_b.execute_query("CREATE SCHEMA IF NOT EXISTS team_b;")

# Both connectors write to a table named 'results' — completely independent
pg_team_a.replace_table(
    pd.DataFrame({"id": [1, 2], "data": ["alpha", "beta"]}),
    "results",
)
pg_team_b.replace_table(
    pd.DataFrame({"id": [10, 20], "data": ["gamma", "delta"]}),
    "results",
)

print("Team A results:")
print(pg_team_a.get_data("SELECT * FROM results;").to_string(index=False))

print("\nTeam B results:")
print(pg_team_b.get_data("SELECT * FROM results;").to_string(index=False))

## Edge Case 7 — Duplicate Rows in the Input DataFrame

**Problem:** If your source data contains duplicate primary-key values within the same DataFrame, the last write wins — but the behaviour may be surprising.

**Best practice:** Deduplicate your DataFrame before upsert when row ordering is meaningful.

In [ ]:
# Two rows share the same id=1 — last one wins in the INSERT ON CONFLICT cycle
dupes = pd.DataFrame({
    "id":    [1, 1, 2],
    "value": ["first", "second", "only"],
})

pg.upsert_data(dupes, "dedup_test", primary_key="id", conflict_strategy="last")
result = pg.get_data("SELECT * FROM dedup_test ORDER BY id;")
print(result)
# id=1 → 'second' (the second row in the DataFrame won)

In [ ]:
# Recommended: deduplicate before upsert to make intent explicit
deduped = dupes.drop_duplicates(subset="id", keep="last")
print("After dedup:")
print(deduped)

## Edge Case 8 — JSONB and Nested Data

**Problem:** Python dicts and lists are not standard SQL types. A naive driver will either refuse them or serialize them as plain TEXT, losing PostgreSQL's JSONB query capabilities.

**What the connector does:** `_generate_dtype_mapping()` detects `dict`/`list` values and maps those columns to `JSONB`, enabling full JSON operators (`->`, `->>`, `@>`) in subsequent queries.

In [ ]:
df = pd.DataFrame({
    "user_id": [1, 2, 3],
    "profile": [
        {"role": "admin", "verified": True},
        {"role": "editor", "verified": False},
        {"role": "viewer", "verified": True, "tags": ["beta"]},
    ],
})

pg.upsert_data(df, "users", primary_key="user_id")

# Query using PostgreSQL JSONB operators
admins = pg.get_data("""
    SELECT user_id, profile->>'role' AS role
    FROM users
    WHERE profile->>'verified' = 'true';
""")
print(admins)

## Cleanup

In [ ]:
for tbl in ["scores", "nulls_demo", "casing_demo", "wide_table", "dedup_test", "users"]:
    pg.execute_query(f"DROP TABLE IF EXISTS {tbl};")

for conn in [pg_team_a, pg_team_b]:
    conn.execute_query("DROP TABLE IF EXISTS results;")
    conn.dispose()

pg.dispose()
print("Done.")

## Summary

| Edge case | What breaks without the connector | How it's handled |
|-----------|-----------------------------------|------------------|
| NumPy scalars | `can't adapt type 'numpy.int64'` | `.item()` call in `_clean_records` |
| NaN / NaT | Stored as string `"nan"` or crash | `pd.isna()` → `None` in `_clean_records` |
| Mixed-case columns | Unmatchable identifiers in DB | `_lower_df_columns()` before all DDL/DML |
| Empty DataFrame | Crash or silent failure | `if df.empty: return` guard at top |
| Wide DataFrames | `ProgrammingError: too many parameters` | `_safe_chunk_size()` splits automatically |
| Multi-schema | Table name collisions across teams | Schema prefix on every SQL statement |
| Duplicate PKs in input | Undefined last-write ordering | Deduplicate with `drop_duplicates` first |
| Dict / list columns | Stored as TEXT, no JSONB operators | Mapped to `JSONB` by `_generate_dtype_mapping` |

**Next:** [Part 5 — Advanced psycopg 3 Features](Part5_Advanced_psycopg3.ipynb)